# Responses API streaming in Jupyter

This notebook shows the compact default transcript, category filtering, silent consumption, and opt-in protocol details. The examples force Web Search and Code Interpreter so their progress events and streamed Python are easy to see.

Install `yhelpers`, then set lowercase `folder_id` and `api_key` environment variables.

In [1]:
import sys
sys.path.append("../../src")
import os

from openai import OpenAI
from yhelpers.responses.streaming import jstream

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
)
model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8"
model = f"gpt://{folder_id}/deepseek-v4-flash"

## Web Search with the compact default

With no flags, `jstream` shows useful activity—such as “Web search”, the query, citations, and the answer—but hides response lifecycle bookkeeping, token usage, and diagnostic JSON.

In [2]:
web_stream = client.responses.create(
    model=model,
    instructions=(
        "You must use web search. Cite the source URL and clearly separate "
        "the verified fact from your explanation."
    ),
    input=(
        "Find the title currently shown on the Python 3 documentation home "
        "page. Then explain in one sentence what that page is for."
    ),
    tools=[{"type": "web_search", "search_context_size": "low"}],
    stream=True,
)
web_response = jstream(web_stream)
print("Response:", web_response.id, web_response.status)

<style>.yhelpers-stream-2b1161f69d0-reasoning ~ *{color:#9ca3af;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-2b1161f69d0-reasoning' style='display:none'></div>

**Reasoning summary**

The user wants me to find the title currently shown on the Python 3 documentation home page. Let me search for this.

<style>.yhelpers-stream-2b1161f69d0-reasoning ~ *{color:#9ca3af;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-2b1161f69d0-reasoning' style='display:none'></div>

**Reasoning summary**

The home page at https://docs.python.org/3/index.html shows title "3.14.7 Documentation". Let me verify this directly. The search result shows the page title is "3.14.7 Documentation". Let me confirm this is current.

The user asks for the title currently shown on the Python 3 documentation home page. According to the search, it's "3.14.7 Documentation".

Let me cite the source URL: https://docs.python.org/3/index.html

<style>.yhelpers-stream-2b1161f69d0-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-2b1161f69d0-text' style='display:none'></div>

Based on my web search, the title currently shown on the Python 3 documentation home page (https://docs.python.org/3/index.html) is **"3.14.7 Documentation"** — this is the official documentation page for the current version of the language (Python 3.14.7).

**Verified fact / source:** The page title is "3.14.7 Documentation." Source: https://docs.python.org/3/index.html (a search snippet also references that "the latest version of the documentation is Python 3.14.7").

**Explanation:** This home page serves as the central entry point to the official Python 3 reference manuals, containing links to the tutorial, language reference, library reference, installation guides, "What's New," and other sections.

Response: d9b19baa-2868-49af-962c-fcf1ba22289e completed


## Code Interpreter with selected categories

This call shows only answer text, Python execution, and errors. Generated Python remains complete and is finalized as a syntax-highlighted `python` fence.

In [3]:
code_stream = client.responses.create(
    model=model,
    instructions=(
        "You must use Code Interpreter. Show the exact Python code, report "
        "the numerical result, and explain the formula in one sentence."
    ),
    input="Calculate the sum of the squares from 1 through 100 with Python.",
    include=["code_interpreter_call.outputs"],
    tools=[
        {
            "type": "code_interpreter",
            "container": {"type": "auto"},
        }
    ],
    stream=True,
)
code_response = jstream(
    code_stream,
    events={"text", "code", "errors"},
)
print("Response:", code_response.id, code_response.status)

<style>.yhelpers-stream-2b114343250-code-interpreter ~ *{color:#7c3aed;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-2b114343250-code-interpreter' style='display:none'></div>

**Code Interpreter**

```python
try:
    # Sum of squares from 1 through 100
    n = 100
    result = sum(i**2 for i in range(1, n+1))
    print("Computing sum of squares from 1 to 100...")
    print(f"result = {result}")
except Exception as e:
    print("Error:", e)
```

<style>.yhelpers-stream-2b114343250-code-interpreter ~ *{color:#7c3aed;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-2b114343250-code-interpreter' style='display:none'></div>

**Python output**

```text
Computing sum of squares from 1 to 100...
result = 338350

```

<style>.yhelpers-stream-2b114343250-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-2b114343250-text' style='display:none'></div>

**Exact Python code:**

```python
n = 100
result = sum(i**2 for i in range(1, n+1))
print(result)  # 338350
```

**Numerical result:** The sum of squares from 1 through 100 is **338350**.

**Formula explanation:** This equals the closed-form sum of squares ∑_{i=1}^{n} i² = n(n+1)(2n+1)/6, which for n = 100 gives 100·101·201/6 = 338350.

Response: 68fc1f72-79eb-4b0b-893a-2ff35b5202c0 completed


## Silent consumption

An empty event collection still consumes the complete stream and returns the final `Response`, but displays nothing while it runs.

In [4]:
silent_stream = client.responses.create(
    model=model,
    input="Reply with exactly: stream consumed",
    stream=True,
)
silent_response = jstream(silent_stream, events=[])
print(silent_response.output_text)

stream consumed


## Full protocol diagnostics (opt in)

`events="all"` includes lifecycle and usage events. `show_details=True` adds event names and bounded JSON payloads. Use this combination when diagnosing SDK or provider behavior; it is intentionally more verbose than the default.

In [4]:
debug_stream = client.responses.create(
    model=model,
    instructions="Use web search once, then answer with one sourced sentence.",
    input="What is the official Python documentation URL?",
    tools=[{"type": "web_search", "search_context_size": "low"}],
    stream=True,
)
debug_response = jstream(
    debug_stream,
    events="all",
    show_details=True,
    max_chars=1000,
)
print("Debug response:", debug_response.id)

<style>.yhelpers-stream-2b1163bec90-reasoning ~ *{color:#9ca3af;font-size:0.875rem;line-height:1.5}}</style><div class='yhelpers-stream-2b1163bec90-reasoning' style='display:none'></div>

**Reasoning summary**

The user is asking about the official Python documentation URL. This is a well-known fact, but let me search anyway since the instructions say to use web search once.

<style>.yhelpers-stream-2b1163bec90-text ~ *{color:#000000;font-size:1.05rem;line-height:1.5}}</style><div class='yhelpers-stream-2b1163bec90-text' style='display:none'></div>

The official Python documentation is available at **https://docs.python.org/** (with the main portal at **https://www.python.org/doc/**), where current and historical language versions are documented.

Debug response: b812fffc-7696-4e2a-93e9-cc007e8cf563


In [ ]:
client.close()